# Silver — Order Items

Enrich line items with category translations; route invalid prices to DLQ.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.transformations.silver_entities as silver_entities_module
importlib.reload(silver_entities_module)

from pyspark.sql import functions as F
from src.transformations.silver_entities import (
    SilverEntitiesConfig, build_silver_order_items, invalid_price_items,
)
from src.quality.dlq import route_to_dlq
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = SilverEntitiesConfig()
print("Loaded from:", silver_entities_module.__file__)

In [ ]:
enriched = build_silver_order_items(spark, config)
invalid = invalid_price_items(enriched)
valid = enriched.join(invalid.select("order_id", "order_item_id"), on=["order_id", "order_item_id"], how="left_anti")

dlq_count = route_to_dlq(
    spark, invalid,
    source_table=config.bronze_order_items,
    failure_reason="invalid_price",
    key_column="order_item_id",
)
valid.write.format("delta").mode("overwrite").saveAsTable(config.silver_order_items)
print("DLQ routed:", dlq_count)
print("Silver order items:", spark.table(config.silver_order_items).count())

In [ ]:
items = spark.table(config.silver_order_items)
unknown_pct = round(100.0 * items.filter(F.col("category_name_en") == "unknown").count() / items.count(), 2)
top_untranslated = (
    items.filter(F.col("category_name_en") == "unknown")
    .groupBy("product_category_name")
    .count()
    .orderBy(F.col("count").desc())
    .limit(3)
)
display(top_untranslated)
print(f"Unknown category %: {unknown_pct}")

In [ ]:
import json

report = {
    "silver_table": config.silver_order_items,
    "dlq_invalid_prices": dlq_count,
    "unknown_category_pct": unknown_pct,
    "top_untranslated": [r.asDict() for r in top_untranslated.collect()],
}
print(json.dumps(report, indent=2))
try:
    save_run_report_to_volume(report, dbutils, "/Volumes/globalmart/metadata/run_reports/silver_order_items_latest.json")
except Exception as exc:
    print("Volume save skipped:", exc)